# PASO 1.3 y 1.4: Validación de Estructura y Organización
## Dataset SIDPOL 2018-2026

Este notebook valida:
- Estructura y columnas clave
- Cobertura de años
- Distritos de Lima Metropolitana incluidos
- Consistencia de UBIGEO
- Tipos de delitos registrados

In [1]:
import pandas as pd
import numpy as np
import os

# Cargar el dataset
archivo = '../data/raw/DATASET_Denuncias_Policiales_Ene 2018 a Julio 2026.csv'
df = pd.read_csv(archivo)

print("✅ Archivo cargado exitosamente")
print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")

✅ Archivo cargado exitosamente
Dimensiones: 369100 filas x 8 columnas


## 1. VALIDACIÓN DE COLUMNAS CLAVE

In [2]:
# Verificar columnas
columnas_esperadas = ['ANIO', 'MES', 'DPTO_HECHO_NEW', 'PROV_HECHO', 'DIST_HECHO', 'UBIGEO_HECHO', 'P_MODALIDADES', 'cantidad']
columnas_actuales = df.columns.tolist()

print("Columnas en el dataset:")
print(f"  {columnas_actuales}")
print()
print("Columnas esperadas:")
print(f"  {columnas_esperadas}")
print()

# Verificar que existan todas las columnas esperadas
columnas_faltantes = [col for col in columnas_esperadas if col not in columnas_actuales]
if columnas_faltantes:
    print(f"⚠️ Columnas FALTANTES: {columnas_faltantes}")
else:
    print("✅ Todas las columnas esperadas están presentes")

Columnas en el dataset:
  ['ANIO', 'MES', 'DPTO_HECHO_NEW', 'PROV_HECHO', 'DIST_HECHO', 'UBIGEO_HECHO', 'P_MODALIDADES', 'cantidad']

Columnas esperadas:
  ['ANIO', 'MES', 'DPTO_HECHO_NEW', 'PROV_HECHO', 'DIST_HECHO', 'UBIGEO_HECHO', 'P_MODALIDADES', 'cantidad']

✅ Todas las columnas esperadas están presentes


## 2. TIPOS DE DATOS

In [3]:
print("Tipos de datos:")
print(df.dtypes)
print()
print("Primeras filas:")
df.head(10)

Tipos de datos:
ANIO              int64
MES               int64
DPTO_HECHO_NEW      str
PROV_HECHO          str
DIST_HECHO          str
UBIGEO_HECHO      int64
P_MODALIDADES       str
cantidad          int64
dtype: object

Primeras filas:


,ANIO,MES,DPTO_HECHO_NEW,PROV_HECHO,DIST_HECHO,UBIGEO_HECHO,P_MODALIDADES,cantidad
0,2018,1,AMAZONAS,BAGUA,ARAMANGO,10202,Otros,4
1,2018,1,AMAZONAS,BAGUA,ARAMANGO,10202,Violencia contra la mujer e integrantes,3
2,2018,1,AMAZONAS,BAGUA,BAGUA,10201,Estafa,2
3,2018,1,AMAZONAS,BAGUA,BAGUA,10201,Hurto,25
4,2018,1,AMAZONAS,BAGUA,BAGUA,10201,Otros,52
5,2018,1,AMAZONAS,BAGUA,BAGUA,10201,Robo,5
6,2018,1,AMAZONAS,BAGUA,BAGUA,10201,Violencia contra la mujer e integrantes,22
7,2018,1,AMAZONAS,BAGUA,COPALLIN,10203,Otros,4
8,2018,1,AMAZONAS,BAGUA,COPALLIN,10203,Violencia contra la mujer e integrantes,1
9,2018,1,AMAZONAS,BAGUA,IMAZA,10205,Hurto,1


## 3. COBERTURA DE AÑOS

In [4]:
años = sorted(df['ANIO'].unique())
print(f"Años cubiertos: {años}")
print(f"Rango: {min(años)} a {max(años)}")
print(f"Total de años: {len(años)}")
print()

# Contar registros por año
print("Registros por año:")
registros_por_año = df.groupby('ANIO').size().sort_index()
print(registros_por_año)

Años cubiertos: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Rango: 2018 a 2026
Total de años: 9

Registros por año:
ANIO
2018    33527
2019    38288
2020    38331
2021    43470
2022    46539
2023    48304
2024    48519
2025    46413
2026    25709
dtype: int64


## 4. COBERTURA GEOGRÁFICA: DEPARTAMENTOS

In [7]:
deptos = sorted(df['DPTO_HECHO_NEW'].unique())
print(f"Departamentos cubiertos ({len(deptos)}):") 
print(deptos)
print()

# Verificar que LIMA METROPOLITANA esté incluida
if 'LIMA METROPOLITANA' in deptos:
    print("✅ LIMA METROPOLITANA está incluida en el dataset")
    
    # Filtrar solo Lima Metropolitana
    df_lima = df[df['DPTO_HECHO_NEW'] == 'LIMA METROPOLITANA'].copy()
    print(f"   Registros en Lima: {len(df_lima)}")
    
    # Provincias en Lima
    provincias_lima = sorted(df_lima['PROV_HECHO'].unique())
    print(f"\n   Provincias en Lima ({len(provincias_lima)}): {provincias_lima}")
else:
    print("❌ LIMA METROPOLITANA NO está en el dataset")

Departamentos cubiertos (26):
['AMAZONAS', 'ANCASH', 'APURIMAC', 'AREQUIPA', 'AYACUCHO', 'CAJAMARCA', 'CUSCO', 'HUANCAVELICA', 'HUANUCO', 'ICA', 'JUNIN', 'LA LIBERTAD', 'LAMBAYEQUE', 'LIMA METROPOLITANA', 'LORETO', 'MADRE DE DIOS', 'MOQUEGUA', 'PASCO', 'PIURA', 'PROV. CONST. DEL CALLAO', 'PUNO', 'REGION LIMA', 'SAN MARTIN', 'TACNA', 'TUMBES', 'UCAYALI']

✅ LIMA METROPOLITANA está incluida en el dataset
   Registros en Lima: 26527

   Provincias en Lima (1): ['LIMA']


## 5. COBERTURA DE DISTRITOS EN LIMA METROPOLITANA

In [8]:
# Distritos de Lima Provincia (Lima Metropolitana tiene 43 distritos)
# Filtrar por Lima Metropolitana
df_lima_full = df[df['DPTO_HECHO_NEW'] == 'LIMA METROPOLITANA'].copy()

distritos_unicos = sorted(df_lima_full['DIST_HECHO'].unique())
print(f"Distritos en Lima Metropolitana ({len(distritos_unicos)}):") 
for i, dist in enumerate(distritos_unicos, 1):
    print(f"  {i:2d}. {dist}")

print(f"\n✅ Total de distritos: {len(distritos_unicos)}")

Distritos en Lima Metropolitana (43):
   1. ANCON
   2. ATE
   3. BARRANCO
   4. BREÑA
   5. CARABAYLLO
   6. CHACLACAYO
   7. CHORRILLOS
   8. CIENEGUILLA
   9. COMAS
  10. EL AGUSTINO
  11. INDEPENDENCIA
  12. JESUS MARIA
  13. LA MOLINA
  14. LA VICTORIA
  15. LIMA
  16. LINCE
  17. LOS OLIVOS
  18. LURIGANCHO - CHOSICA
  19. LURIN
  20. MAGDALENA DEL MAR
  21. MIRAFLORES
  22. PACHACAMAC
  23. PUCUSANA
  24. PUEBLO LIBRE
  25. PUENTE PIEDRA
  26. PUNTA HERMOSA
  27. PUNTA NEGRA
  28. RIMAC
  29. SAN BARTOLO
  30. SAN BORJA
  31. SAN ISIDRO
  32. SAN JUAN DE LURIGANCHO
  33. SAN JUAN DE MIRAFLORES
  34. SAN LUIS
  35. SAN MARTIN DE PORRES
  36. SAN MIGUEL
  37. SANTA ANITA
  38. SANTA MARIA DEL MAR
  39. SANTA ROSA
  40. SANTIAGO DE SURCO
  41. SURQUILLO
  42. VILLA EL SALVADOR
  43. VILLA MARIA DEL TRIUNFO

✅ Total de distritos: 43


## 6. VALIDACIÓN DE UBIGEO

In [9]:
# Revisar UBIGEOs en Lima
ubigeos_lima = df_lima_full[['DIST_HECHO', 'UBIGEO_HECHO']].drop_duplicates().sort_values('UBIGEO_HECHO')

print("UBIGEO por Distrito en Lima Metropolitana:")
print(ubigeos_lima.to_string(index=False))
print()

# Validar formato de UBIGEO
ubigeos_unicos = df_lima_full['UBIGEO_HECHO'].unique()
ubigeos_validos = all(len(str(ubigeo)) == 6 for ubigeo in ubigeos_unicos)

if ubigeos_validos:
    print("✅ Todos los UBIGEO tienen 6 dígitos (formato válido)")
else:
    print("⚠️ Algunos UBIGEO no tienen 6 dígitos")
    print(f"   UBIGEO únicos: {sorted([str(u) for u in ubigeos_unicos])}")

UBIGEO por Distrito en Lima Metropolitana:
             DIST_HECHO  UBIGEO_HECHO
                   LIMA        150101
                  ANCON        150102
                    ATE        150103
               BARRANCO        150104
                  BREÑA        150105
             CARABAYLLO        150106
             CHACLACAYO        150107
             CHORRILLOS        150108
            CIENEGUILLA        150109
                  COMAS        150110
            EL AGUSTINO        150111
          INDEPENDENCIA        150112
            JESUS MARIA        150113
              LA MOLINA        150114
            LA VICTORIA        150115
                  LINCE        150116
             LOS OLIVOS        150117
   LURIGANCHO - CHOSICA        150118
                  LURIN        150119
      MAGDALENA DEL MAR        150120
           PUEBLO LIBRE        150121
             MIRAFLORES        150122
             PACHACAMAC        150123
               PUCUSANA        150124
       

## 7. TIPOS DE DELITOS / MODALIDADES

In [11]:
print("Valores nulos en el dataset:")
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else "✅ No hay valores nulos")
print()

print("Valores nulos en Lima Metropolitana:")
nulos_lima = df_lima_full.isnull().sum()
print(nulos_lima[nulos_lima > 0] if nulos_lima.sum() > 0 else "✅ No hay valores nulos en Lima")

Valores nulos en el dataset:
✅ No hay valores nulos

Valores nulos en Lima Metropolitana:
✅ No hay valores nulos en Lima


## 8. VALIDACIÓN DE VALORES NULOS

In [14]:
modalidades = sorted(df['P_MODALIDADES'].unique())
print(f"Tipos de Modalidades ({len(modalidades)}):")
for i, mod in enumerate(modalidades, 1):
    print(f"  {i:2d}. {mod}")

print()
print("Registros por Modalidad (Top 20):")
modalidades_count = df['P_MODALIDADES'].value_counts().head(20)
print(modalidades_count)

Tipos de Modalidades (7):
   1. Estafa
   2. Extorsión
   3. Hurto
   4. Otros
   5. Robo
   6. Secuestro
   7. Violencia contra la mujer e integrantes

Registros por Modalidad (Top 20):
P_MODALIDADES
Otros                                      107429
Violencia contra la mujer e integrantes     94941
Hurto                                       67438
Robo                                        41751
Estafa                                      31004
Extorsión                                   19527
Secuestro                                    7010
Name: count, dtype: int64


In [13]:
print("Valores nulos en el dataset:")
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else "✅ No hay valores nulos")
print()

print("Valores nulos en Lima:")
nulos_lima = df_lima.isnull().sum()
print(nulos_lima[nulos_lima > 0] if nulos_lima.sum() > 0 else "✅ No hay valores nulos en Lima")

Valores nulos en el dataset:
✅ No hay valores nulos

Valores nulos en Lima:
✅ No hay valores nulos en Lima


## 9. RESUMEN FINAL

In [15]:
print("="*70)
print("RESUMEN DE VALIDACIÓN - PASO 1.3 y 1.4")
print("="*70)
print()
print(f"✅ Columnas clave presentes: {len(columnas_actuales)} columnas")
print(f"✅ Años cubiertos: {min(años)} a {max(años)} ({len(años)} años)")
print(f"✅ Departamentos: {len(deptos)}")
print(f"✅ Lima incluida: SÍ")
print(f"✅ Distritos en Lima Provincia: {len(distritos_unicos)}")
print(f"✅ UBIGEOs únicos: {len(ubigeos_unicos)}")
print(f"✅ Modalidades de delitos: {len(modalidades)}")
print(f"✅ Total de registros: {len(df):,}")
print(f"✅ Registros en Lima: {len(df_lima):,}")
print()
print("="*70)
print("ESTADO: ✅ VALIDACIÓN EXITOSA - Listo para Fase 2 (Limpieza)")
print("="*70)

RESUMEN DE VALIDACIÓN - PASO 1.3 y 1.4

✅ Columnas clave presentes: 8 columnas
✅ Años cubiertos: 2018 a 2026 (9 años)
✅ Departamentos: 26
✅ Lima incluida: SÍ
✅ Distritos en Lima Provincia: 43
✅ UBIGEOs únicos: 43
✅ Modalidades de delitos: 7
✅ Total de registros: 369,100
✅ Registros en Lima: 26,527

ESTADO: ✅ VALIDACIÓN EXITOSA - Listo para Fase 2 (Limpieza)
